In [ ]:
# 1. Environment setup, path monkey-patch, and CUDA check
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import pathlib
# Apply path monkey-patch immediately to avoid encoding issues with third-party libraries on Windows
if not getattr(pathlib.Path.read_text, "__patched__", False):
    orig_read_text = pathlib.Path.read_text
    def patched_read_text(self, encoding=None, errors=None):
        return orig_read_text(self, encoding='utf-8' if encoding is None else encoding, errors=errors)
    patched_read_text.__patched__ = True
    pathlib.Path.read_text = patched_read_text

# IMPORTANT: datasets must be imported before torch/transformers to avoid silent DLL crash on Windows!
from datasets import Dataset
import torch
print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is NOT available. Running SFT training will be extremely slow on CPU.")

In [ ]:
# 2. Load and preprocess NL -> FOL translation datasets
import json
import os

def load_translation_dataset(logic_path: str, folio_path: str) -> list[dict]:
    samples = []
    seen_premises = set()
    
    # 1. Load logic_based.json
    if os.path.exists(logic_path):
        with open(logic_path, "r", encoding="utf-8") as f:
            logic_data = json.load(f)
        for item in logic_data:
            nl_list = item.get("premises-NL", [])
            fol_list = item.get("premises-FOL", [])
            if not nl_list or not fol_list or len(nl_list) != len(fol_list):
                continue
            
            # Use NL premises string for deduplication
            nl_serialized = "\n".join(nl_list)
            if nl_serialized in seen_premises:
                continue
            seen_premises.add(nl_serialized)
            
            samples.append({
                "premises-NL": nl_list,
                "premises-FOL": fol_list
            })
        print(f"Loaded {len(samples)} unique instances from logic_based.json")
        
    # 2. Load folio-train.json
    logic_count = len(samples)
    if os.path.exists(folio_path):
        with open(folio_path, "r", encoding="utf-8") as f:
            folio_data = json.load(f)
        for item in folio_data:
            nl_list = item.get("premises-NL", [])
            fol_list = item.get("premises-FOL", [])
            if not nl_list or not fol_list or len(nl_list) != len(fol_list):
                continue
            
            nl_serialized = "\n".join(nl_list)
            if nl_serialized in seen_premises:
                continue
            seen_premises.add(nl_serialized)
            
            samples.append({
                "premises-NL": nl_list,
                "premises-FOL": fol_list
            })
        print(f"Loaded {len(samples) - logic_count} unique instances from folio-train.json")
        
    print(f"Total unique translation instances: {len(samples)}")
    return samples

# Find project root directory and define unified results folder
project_root = "."
for possible_root in [".", "..", "../..", "../../../", "../../../.."]:
    if os.path.exists(os.path.join(possible_root, "pyproject.toml")):
        project_root = os.path.abspath(possible_root)
        break

results_dir = os.path.join(project_root, "results")
os.makedirs(results_dir, exist_ok=True)
print(f"Unified results folder: {results_dir}")

# Resolve dataset paths relative to project_root
logic_path = os.path.join(project_root, "data/processed/logic_based.json")
folio_path = os.path.join(project_root, "data/processed/folio-train.json")

raw_samples = load_translation_dataset(logic_path, folio_path)

# Format to conversational messages for chat template SFT
formatted_samples = []
for item in raw_samples:
    nl_list = item["premises-NL"]
    fol_list = item["premises-FOL"]
    
    # Format NL premises
    nl_content = "Translate the following natural language premises into first-order logic (FOL) formulas:\n"
    for i, nl in enumerate(nl_list, start=1):
        nl_content += f"{i}. {nl}\n"
        
    # Format FOL premises
    fol_content = ""
    for i, fol in enumerate(fol_list, start=1):
        fol_content += f"{i}. {fol}\n"
        
    system_prompt = "You are a logical reasoning assistant. Translate the given natural language premises into first-order logic (FOL) formulas."
    
    formatted_samples.append({
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": nl_content.strip()},
            {"role": "assistant", "content": fol_content.strip()}
        ]
    })

# Print a formatted sample to inspect
print("\n--- Sample Formatted Conversation ---")
for msg in formatted_samples[0]["messages"]:
    print(f"[{msg['role'].upper()}]:\n{msg['content']}\n")

In [ ]:
# 3. Model & Tokenizer Initialization
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

model_id = "Qwen/Qwen3-8B"
print(f"Initializing model and tokenizer for {model_id}...")

# Configure 4-bit quantization for VRAM efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer: load from results_dir if it exists, otherwise model_id
if os.path.exists(os.path.join(results_dir, "tokenizer_config.json")):
    print(f"Loading existing tokenizer from {results_dir}...")
    tokenizer = AutoTokenizer.from_pretrained(results_dir, trust_remote_code=True)
else:
    print(f"Loading base tokenizer from {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device_map = "auto" if torch.cuda.is_available() else None
print(f"Loading base model to device_map: {device_map}")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map=device_map,
    trust_remote_code=True,
    attn_implementation="sdpa"
)

# Disable cache to prevent warnings and save memory during training
base_model.config.use_cache = False

# Prepare for QLoRA training
if torch.cuda.is_available():
    base_model = prepare_model_for_kbit_training(base_model)

# Standard target modules for Qwen attention and MLP layers
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Load existing adapter if it exists, otherwise initialize new one
adapter_config_path = os.path.join(results_dir, "adapter_config.json")
if os.path.exists(adapter_config_path):
    print(f"Found existing trained adapter at {results_dir}. Loading adapter to resume training...")
    model = PeftModel.from_pretrained(base_model, results_dir, is_trainable=True)
else:
    print(f"No existing trained adapter found. Initializing a new PEFT adapter...")
    model = get_peft_model(base_model, peft_config)

model.print_trainable_parameters()

In [ ]:
# 4. Prepare HF Dataset and Train/Val Split
from datasets import Dataset

dataset = Dataset.from_list(formatted_samples)

# Apply chat template
def apply_template(batch):
    texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(apply_template, batched=True, remove_columns=["messages"])

# Split into Train (90%) and Val (10%)
dataset = dataset.shuffle(seed=42)
split_dataset = dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"Dataset mapped and split. Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

In [ ]:
# 5. Configure SFT Training and Execute Trainer
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import glob

class LossLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                print(f"Step {state.global_step}: training_loss = {logs['loss']:.6f}")
            if "eval_loss" in logs:
                print(f"Step {state.global_step}: validation_loss = {logs['eval_loss']:.6f}")

output_dir = results_dir
max_length = 512  # Optimal for RTX 4060 VRAM, preventing Windows memory paging

sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,  # Effective batch size = 8
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    logging_steps=1,
    logging_first_step=True,
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    gradient_checkpointing=True if torch.cuda.is_available() else False,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    per_device_eval_batch_size=1,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    dataset_text_field="text",
    max_length=max_length
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=sft_config,
    callbacks=[LossLoggingCallback()]
)

# Automatically resume from the latest checkpoint if it exists
checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
resume_path = None
if checkpoints:
    # Sort checkpoints by their step number
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    resume_path = checkpoints[-1]
    print(f"Found checkpoints. Resuming training from: {resume_path}")
else:
    print("No checkpoints found. Starting training from scratch.")

print("Starting SFT training loop...")
trainer.train(resume_from_checkpoint=resume_path)

# Save final adapter weights (this saves the best model because load_best_model_at_end=True loads the best weights)
print(f"Saving best validation fine-tuned adapter and tokenizer to {output_dir}...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Fine-tuning pipeline completed successfully!")

In [ ]:
# 6. Test Inference with Fine-Tuned Model
import torch
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Cleaning up training resources from GPU memory before testing inference...")
# Delete trainer to free up optimizer states and training gradients
if "trainer" in globals():
    del trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Note: Loading the model twice on a GPU with limited VRAM (like an 8GB RTX 4060)
# will cause an Out-Of-Memory (OOM) error. Therefore, we reuse the already loaded
# `model` directly in this session.
#
# If you are in a fresh Jupyter session and want to load the trained adapter from disk,
# you can uncomment the code below:
"""
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

adapter_path = results_dir
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, adapter_path)
"""

print("Reusing the trained model for testing inference...")
model.eval()

# Sample test prompt
test_nl_premises = [
    "All humans are mortal.",
    "Socrates is a human."
]

test_nl_content = "Translate the following natural language premises into first-order logic (FOL) formulas:\n"
for i, nl in enumerate(test_nl_premises, start=1):
    test_nl_content += f"{i}. {nl}\n"

system_prompt = "You are a logical reasoning assistant. Translate the given natural language premises into first-order logic (FOL) formulas."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_nl_content.strip()}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128, eos_token_id=tokenizer.eos_token_id)

response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- NL Premises ---")
print(test_nl_content.strip())
print("\n--- Model Generated FOL Translation (Expected) ---")
print(response_text)
print("--------------------------------------------------")